In [12]:
import sys
!{sys.executable} -m pip install --user \
    torch \
    scikit-image \
    scikit-learn \
    scipy \
    matplotlib \
    pandas \
    tqdm \
    seaborn

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [13]:
import numpy as np
import torch
import os
import scipy.io as sio
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader,random_split
from sklearn import preprocessing
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
import torch.nn as nn
from torch.nn import functional as F
import gc
from optparse import OptionParser
from tqdm import tqdm
import time
import csv
import pandas as pd
# from main_train import *
# from main_test import *
from Data_loader import *
#from architectures.Network import *
from metrics_util import * 
#from architectures.FNO3dTo2d import FNO3dTo2d
#from architectures.LTAE_Net import MRE_UTAE
#from architectures.UpdatedNetwork import Net
#from architectures.SimpleTFusionUNet3D import SimpleTFusionUNet3D
#from architectures.UNO import UNOWithTemporalReduction
#from architectures.arch_utils import *
from train_functions import *
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
# ---- Dynamic RUN_IDS extraction from Slurm report files ----
import re
import glob

def extract_run_ids_from_reports(slurm_dir: str) -> dict:
    """
    Scan Report-*.out files in slurm_dir and extract the last run_YYYYMMDD_HHMMSS
    identifier from each file.
    
    Returns:
        dict mapping report filename -> run_id (or None if not found)
    """
    run_pattern = re.compile(r'(run_\d{8}_\d{6})')
    results = {}
    
    report_files = sorted(glob.glob(os.path.join(slurm_dir, 'Report-*.out')))
    
    for report_path in report_files:
        basename = os.path.basename(report_path)
        with open(report_path, 'r') as f:
            content = f.read()
        matches = run_pattern.findall(content)
        results[basename] = matches[-1] if matches else None
    
    return results

SLURM_DIR = os.path.join(os.getcwd(), 'Slurm_Training')
report_to_run = extract_run_ids_from_reports(SLURM_DIR)

# Filter out None (direct inversion) and deduplicate
RUN_IDS = sorted(set(rid for rid in report_to_run.values() if rid is not None))

print(f'Found {len(report_to_run)} report files in {SLURM_DIR}:')
print('-' * 70)
for report, run_id in sorted(report_to_run.items()):
    status = run_id if run_id else '(no ML run - direct inversion)'
    print(f'  {report:45s} -> {status}')
print('-' * 70)
print(f'\nUnique RUN_IDS ({len(RUN_IDS)}):')
for rid in RUN_IDS:
    print(f'  {rid}')


Found 12 report files in /storage/project/r-jueda3-0/jueda3/MRE_OAGH_test/MinimalCode_3_5_2026/root/Slurm_Training:
----------------------------------------------------------------------
  Report-direct_inv-6262342.out                 -> (no ML run - direct inversion)
  Report-fixed_het-6262343.out                  -> run_20260403_201308
  Report-fixed_hom-6262344.out                  -> run_20260403_204633
  Report-het_gradnorm-6262345.out               -> run_20260403_205014
  Report-het_oagh-6262347.out                   -> run_20260403_224731
  Report-het_oagh_c-6262346.out                 -> run_20260403_224731
  Report-het_pcgrad-6262348.out                 -> run_20260403_232845
  Report-hom_gradnorm-6262349.out               -> run_20260404_000306
  Report-hom_oagh-6262351.out                   -> run_20260404_000918
  Report-hom_oagh_c-6262350.out                 -> run_20260404_000918
  Report-hom_pcgrad-6262352.out                 -> run_20260404_004250
  Report-mse_only-626

In [15]:
import os

base_dir = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet"
target_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if "weights.pth" in file  and ("bs64" in root or "bs64" in file):
            target_files.append(os.path.join(root, file))

print(f"Found {len(target_files)} matching weights.pth files:")
for f in target_files:
    print(f)


Found 11 matching weights.pth files:
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/mse/lam_data1_0/lam_phys0_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/gradnorm/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/pcgrad/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_

In [16]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def plot_per_sample_metrics(df, metrics=None, plot_type="box", figsize=(12,6), clip=None, dataset_filter=None, show=False):
    """
    Plot distributions of per-sample metrics across models and SNRs, with optional clipping.
    
    Args:
        df (pd.DataFrame): Long-format per-sample metrics DataFrame.
        metrics (list): List of metrics to plot. Default: all numeric metrics.
        plot_type (str): 'box' or 'violin'.
        figsize (tuple): Figure size.
        clip (tuple or None): Optional (min, max) to clip y-axis values.
        dataset_filter (str or None): Filter to specific dataset (e.g., "Main_Test", "Inclusion_R0.3125cm")
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    if metrics is None:
        # Automatically pick numeric columns
        metrics = [c for c in plot_df.columns if c not in ["index", "snr_db", "model", "dataset"]]
    
    for metric in metrics:
        plt.figure(figsize=figsize)
        
        # Clip metric values if requested
        if clip is not None:
            min_val, max_val = clip
            plot_df[metric] = plot_df[metric].clip(lower=min_val, upper=max_val)
        
        if plot_type == "box":
            sns.boxplot(
                data=plot_df,
                x="snr_db",
                y=metric,
                hue="model",
                palette="Set2"
            )
        elif plot_type == "violin":
            sns.violinplot(
                data=plot_df,
                x="snr_db",
                y=metric,
                hue="model",
                split=False,  # Changed to False for multiple models
                palette="Set2"
            )
        else:
            raise ValueError("plot_type must be 'box' or 'violin'")
        
        title = f"Per-sample {metric} distribution"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel(metric)
        plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        if show:
            plt.show()


def plot_metric_heatmap(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.75, dataset_filter=None, show=False):
    """
    Plot heatmaps for a metric across models and SNR levels.
    
    Args:
        df: DataFrame from run_full_evaluation() (aggregate)
        metric_prefix: "mae", "rmse", "ssim", or "cnr"
        clip_outliers: if True, heatmap color scale will ignore extreme values
        lower_q, upper_q: quantiles for clipping when clip_outliers=True
        dataset_filter: Filter to specific dataset
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    # Check if metric has _k and _mu variants
    has_k = f"{metric_prefix}_k" in plot_df.columns
    has_mu = f"{metric_prefix}_mu" in plot_df.columns
    is_cnr = metric_prefix == "cnr" and "cnr" in plot_df.columns
    
    if is_cnr:
        # CNR only has one column
        pivot = plot_df.pivot(index="model", columns="snr_db", values="cnr")
        
        if clip_outliers:
            vmin = pivot.stack().quantile(lower_q)
            vmax = pivot.stack().quantile(upper_q)
        else:
            vmin, vmax = pivot.min().min(), pivot.max().max()
        
        plt.figure(figsize=(8,6))
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=vmin, vmax=vmax)
        title = f"CNR"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel("Model")
        plt.tight_layout()
        plt.show()
        
    elif has_k and has_mu:
        pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_k")
        pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_mu")
        
        # Determine vmin/vmax if clipping
        if clip_outliers:
            k_lower = pivot_k.stack().quantile(lower_q)
            k_upper = pivot_k.stack().quantile(upper_q)
            mu_lower = pivot_mu.stack().quantile(lower_q)
            mu_upper = pivot_mu.stack().quantile(upper_q)
        else:
            k_lower, k_upper = pivot_k.min().min(), pivot_k.max().max()
            mu_lower, mu_upper = pivot_mu.min().min(), pivot_mu.max().max()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[0], cmap="magma",
                    vmin=k_lower, vmax=k_upper)
        axes[0].set_title(f"(k) {metric_prefix.upper()}")
        axes[0].set_xlabel("SNR (dB)")
        axes[0].set_ylabel("Model")
        
        sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[1], cmap="magma",
                    vmin=mu_lower, vmax=mu_upper)
        axes[1].set_title(f"(mu) {metric_prefix.upper()}")
        axes[1].set_xlabel("SNR (dB)")
        axes[1].set_ylabel("")
        
        if dataset_filter:
            fig.suptitle(dataset_filter, fontsize=14, y=1.02)
        
        plt.tight_layout()
        if show:
            plt.show()


def plot_metric_vs_snr(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.80, dataset_filter=None, show=False):
    """
    Plot metric vs SNR for each model.
    
    Args:
        df: DataFrame output from run_full_evaluation() (aggregate)
        metric_prefix: "mae", "rmse", "ssim", or "cnr"
        clip_outliers: if True, y-axis will ignore extreme values based on quantiles
        lower_q, upper_q: quantiles to define the y-axis range when clipping
        dataset_filter: Filter to specific dataset
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    # Convert snr_db to numeric for proper sorting and plotting
    snr_mapping = {"clean": 999}
    plot_df["snr_numeric"] = plot_df["snr_db"].apply(
        lambda x: snr_mapping.get(x, float(x)) if x != "clean" else 999
    )
    plot_df = plot_df.sort_values("snr_numeric")
    
    # Create x-axis labels (keep original labels)
    plot_df["snr_label"] = plot_df["snr_db"].apply(
        lambda x: "Clean" if x == "clean" else str(x)
    )
    
    is_cnr = metric_prefix == "cnr" and "cnr" in plot_df.columns
    
    if is_cnr:
        plt.figure(figsize=(8, 5))
        sns.lineplot(
            data=plot_df,
            x="snr_numeric",  # Use numeric for plotting
            y="cnr",
            hue="model",
            marker="o",
            style="model"
        )
        
        # Fix x-axis labels
        unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
        plt.xticks(unique_snr["snr_numeric"], unique_snr["snr_label"])
        
        title = "CNR vs SNR"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel("CNR")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.grid(True)
        
        if clip_outliers:
            lower = plot_df["cnr"].quantile(lower_q)
            upper = plot_df["cnr"].quantile(upper_q)
            plt.ylim(lower, upper)
        
        plt.tight_layout()
        plt.show()
    else:
        for mtype in ["k", "mu"]:
            metric_col = f"{metric_prefix}_{mtype}"
            if metric_col not in plot_df.columns:
                continue
                
            plt.figure(figsize=(8, 5))
            sns.lineplot(
                data=plot_df,
                x="snr_numeric",  # Use numeric for plotting
                y=metric_col,
                hue="model",
                marker="o",
                style="model"
            )
            
            # Fix x-axis labels
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            plt.xticks(unique_snr["snr_numeric"], unique_snr["snr_label"])
            
            title = f"({mtype}) {metric_prefix.upper()} vs SNR"
            if dataset_filter:
                title += f" - {dataset_filter}"
            plt.title(title)
            plt.xlabel("SNR (dB)")
            plt.ylabel(metric_prefix.upper())
            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            plt.grid(True)
            
            if clip_outliers:
                lower = plot_df[metric_col].quantile(lower_q)
                upper = plot_df[metric_col].quantile(upper_q)
                plt.ylim(lower, upper)
            
            plt.tight_layout()
            if show:
                plt.show()


def highlight_best_per_snr_dataset(df):
    """
    Bold the best metrics per SNR and dataset in aggregate DataFrame.
    Returns a styled DataFrame.
    
    For MAE/RMSE: best = min (bold green)
    For SSIM/CNR: best = max (bold green)
    """
    # Identify metric columns
    numeric_cols = df.select_dtypes(include='number').columns
    mae_rmse_cols = [c for c in numeric_cols if 'mae' in c.lower() or 'rmse' in c.lower()]
    ssim_cnr_cols = [c for c in numeric_cols if 'ssim' in c.lower() or 'cnr' in c.lower()]
    
    def apply_highlighting(data):
        """Apply styles to entire DataFrame at once."""
        # Create empty style DataFrame
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        
        # Group by dataset and snr_db
        for (dataset, snr), group_idx in data.groupby(['dataset', 'snr_db']).groups.items():
            # Get the subset of data for this group
            group_data = data.loc[group_idx]
            
            # Highlight best for MAE/RMSE (minimum)
            for col in mae_rmse_cols:
                if col in group_data.columns and not group_data[col].isna().all():
                    best_idx = group_data[col].idxmin()
                    styles.loc[best_idx, col] = 'font-weight: bold; color: green'
            A
            # Highlight best for SSIM/CNR (maximum)
            for col in ssim_cnr_cols:
                if col in group_data.columns and not group_data[col].isna().all():
                    best_idx = group_data[col].idxmax()
                    styles.loc[best_idx, col] = 'font-weight: bold; color: green'
        
        return styles
    
    return df.style.apply(apply_highlighting, axis=None)


def display_best_metrics_table(df, dataset_filter=None, snr_filter=None):
    """
    Display a clean summary table showing only the best model per metric.
    
    Args:
        df: Aggregate metrics DataFrame
        dataset_filter: Filter to specific dataset
        snr_filter: Filter to specific SNR value
    """
    filtered_df = df.copy()
    
    if dataset_filter:
        filtered_df = filtered_df[filtered_df['dataset'] == dataset_filter]
    if snr_filter is not None:
        filtered_df = filtered_df[filtered_df['snr_db'] == snr_filter]
    
    # Get metric columns
    numeric_cols = filtered_df.select_dtypes(include='number').columns
    mae_rmse_cols = [c for c in numeric_cols if 'mae' in c.lower() or 'rmse' in c.lower()]
    ssim_cnr_cols = [c for c in numeric_cols if 'ssim' in c.lower() or 'cnr' in c.lower()]
    
    results = []
    
    for (dataset, snr), group in filtered_df.groupby(['dataset', 'snr_db']):
        row = {'dataset': dataset, 'snr_db': snr}
        
        # Find best model for each metric (skip if all NaN)
        for col in mae_rmse_cols:
            if col in group.columns and not group[col].isna().all():
                best_idx = group[col].idxmin()
                if pd.notna(best_idx):  # Check if index is valid
                    best_model = group.loc[best_idx, 'model']
                    best_value = group.loc[best_idx, col]
                    row[f'{col}_model'] = best_model
                    row[f'{col}_value'] = f'{best_value:.4f}'
        
        for col in ssim_cnr_cols:
            if col in group.columns and not group[col].isna().all():
                best_idx = group[col].idxmax()
                if pd.notna(best_idx):  # Check if index is valid
                    best_model = group.loc[best_idx, 'model']
                    best_value = group.loc[best_idx, col]
                    row[f'{col}_model'] = best_model
                    row[f'{col}_value'] = f'{best_value:.4f}'
        
        results.append(row)
    
    summary_df = pd.DataFrame(results)
    return summary_df

In [ ]:
import os
import re
import torch
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional
from Data_loader import get_dataloader_for_test
from train_functions import wave_number_to_shear_stiffness, setup_model
from evaluation.metrics import compute_mae, compute_rmse, compute_ssim, compute_cnr, get_masks
from datetime import datetime
from pathlib import Path
import json


class ModelEvaluator:
    """Comprehensive model evaluation across datasets and SNR levels."""
    
    def __init__(
        self,
        root_dir: str = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs",
        device: str = "cuda"
    ):
        self.root = Path(root_dir)
        self.device = device
        
    def find_weights_by_runs(self, run_ids: List[str]) -> List[str]:
        """
        Find all weights.pth matching specific run IDs.
        
        Args:
            run_ids: List of run IDs like ["run_20260223_133208", "run_20260223_140110"]
        """
        # Extract just the numeric parts if full run_YYYYMMDD_HHMMSS provided
        numeric_ids = []
        for rid in run_ids:
            if rid.startswith("run_"):
                # Extract YYYYMMDD_HHMMSS part
                parts = rid.split("_")
                if len(parts) >= 3:
                    numeric_ids.append(parts[-1])  # Get HHMMSS
                else:
                    numeric_ids.append(rid)
            else:
                numeric_ids.append(rid)
        
        # Build regex pattern
        run_pat = re.compile(r"run_\d{8}_(" + "|".join(numeric_ids) + r")")
        
        def bs64_has_matching_tb(bs64_dir: Path) -> bool:
            """Check if bs64 directory contains matching tensorboard logs."""
            # Fast path: look for tfevents files with matching run id
            for p in bs64_dir.rglob("*"):
                s = str(p)
                if run_pat.search(s) and ("tfevents" in p.name or "events.out.tfevents" in p.name):
                    return True
            
            # Fallback: any path under bs64 contains the run id
            for p in bs64_dir.rglob("*"):
                if run_pat.search(str(p)):
                    return True
            
            return False
        
        matched_weights = []
        
        # Find all weights.pth
        for w in self.root.rglob("weights.pth"):
            bs64_dir = w.parent  # .../bs64
            if bs64_dir.name != "bs64":
                # If weights live deeper, you can remove this guard
                continue
            
            if bs64_has_matching_tb(bs64_dir):
                matched_weights.append(str(w))
        
        print(f"Matched weights.pth files: {len(matched_weights)}")
        for p in matched_weights:
            print(f"  {p}")
        
        return matched_weights
    
    def find_weights_by_filter(self, filter_dict: Dict[str, any]) -> List[str]:
        """
        Find weights.pth based on flexible filters.
        
        filter_dict examples:
            {"loss_type": "mse"}
            {"loss_type": "residual", "lambda_physics": 0.025}
            {"loss_type": ["mse", "residual"]}
        """
        matched_weights = []
        
        for w in self.root.rglob("weights.pth"):
            # Load checkpoint to check metadata
            try:
                ckpt = torch.load(w, map_location="cpu")
                
                # Check all filter conditions
                match = True
                for key, value in filter_dict.items():
                    if key not in ckpt:
                        match = False
                        break
                    
                    # Handle list of acceptable values
                    if isinstance(value, list):
                        if ckpt[key] not in value:
                            match = False
                            break
                    else:
                        if ckpt[key] != value:
                            match = False
                            break
                
                if match:
                    matched_weights.append(str(w))
                    
            except Exception as e:
                print(f"Warning: Could not load {w}: {e}")
                continue
        
        print(f"Found {len(matched_weights)} weights matching filter: {filter_dict}")
        return matched_weights
    
    def load_models(self, weight_paths: List[str]) -> Dict[str, torch.nn.Module]:
        """Load models from weight paths."""
        models = {}
        
        for pth_path in weight_paths:
            # Create key based on relative path from root
            path_obj = Path(pth_path)
            key = os.path.relpath(os.path.dirname(pth_path), self.root)
            
            # Load checkpoint
            ckpt = torch.load(pth_path, map_location=self.device)
            
            # Setup model
            model = setup_model("FDTDNet")[0]
            model.load_state_dict(ckpt["state_dict"])
            model.eval()
            model.to(self.device)
            
            models[key] = model
        
        print(f"\nLoaded {len(models)} models:")
        for k in models.keys():
            print(f"  - {k}")
        
        return models
    
    def evaluate_per_sample_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> pd.DataFrame:
        """Evaluate model per sample and return DataFrame."""
        model.eval()
        records = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                # Permute wave
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                # Predict
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                # Metrics per sample
                batch_size = wave.shape[0]
                for b in range(batch_size):
                    record = {
                        "index": int(index[b]),
                        "mae_k": compute_mae(pred_k[b], k_gt[b]),
                        "rmse_k": compute_rmse(pred_k[b], k_gt[b]),
                        "ssim_k": compute_ssim(pred_k[b], k_gt[b]),
                        "mae_mu": compute_mae(pred_mu[b], mu[b]),
                        "rmse_mu": compute_rmse(pred_mu[b], mu[b]),
                        "ssim_mu": compute_ssim(pred_mu[b], mu[b])
                    }
                    
                    # CNR if inclusions
                    if has_inclusions:
                        inc_mask, bg_mask = get_masks(mu[b])
                        cnr_value = compute_cnr(
                        pred_mu[b].squeeze(),
                        inc_mask,
                        bg_mask
                        )
                        record["cnr"] = cnr_value.item() if torch.is_tensor(cnr_value) else cnr_value
                    records.append(record)  # ← This line was missing!

        
        return pd.DataFrame(records)
    
    def evaluate_aggregate_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> Dict:
        """Evaluate model and return aggregated metrics."""
        model.eval()
        metrics = {
            "mae_k": [], "rmse_k": [], "ssim_k": [],
            "mae_mu": [], "rmse_mu": [], "ssim_mu": []
        }
        if has_inclusions:
            metrics["cnr"] = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                metrics["mae_k"].append(compute_mae(pred_k, k_gt))
                metrics["rmse_k"].append(compute_rmse(pred_k, k_gt))
                metrics["ssim_k"].append(compute_ssim(pred_k, k_gt))
                metrics["mae_mu"].append(compute_mae(pred_mu, mu))
                metrics["rmse_mu"].append(compute_rmse(pred_mu, mu))
                metrics["ssim_mu"].append(compute_ssim(pred_mu, mu))
                
                if has_inclusions:
                    inc_mask, bg_mask = get_masks(mu)
                    metrics["cnr"].append(compute_cnr(pred_mu.squeeze(), inc_mask, bg_mask))
        
        return {k: (torch.tensor(v).mean().item() if v else 0.0) for k, v in metrics.items()}
    
    def run_full_evaluation(
        self,
        models: Dict[str, torch.nn.Module],
        test_configs: List[Dict],
        per_sample: bool = False
    ) -> pd.DataFrame:
        """
        Run evaluation across models and test configurations.
        
        test_configs: List of dicts with keys:
            - test_dir: path to test data
            - snr_levels: list of SNR values (None for clean)
            - has_inclusions: bool
            - dataset_name: str (for labeling)
        """
        all_rows = []
        
        for config in test_configs:
            test_dir = config["test_dir"]
            snr_levels = config["snr_levels"]
            has_inclusions = config.get("has_inclusions", False)
            dataset_name = config.get("dataset_name", "Unknown")
            
            print(f"\n{'='*60}")
            print(f"Evaluating on: {dataset_name}")
            print(f"{'='*60}")
            
            for model_name, model in models.items():
                for snr in snr_levels:
                    loader = get_dataloader_for_test(
                        test_dir,
                        batch_size=1,
                        snr_db=snr
                    )
                    
                    if per_sample:
                        df = self.evaluate_per_sample_metrics(
                            model, loader, has_inclusions
                        )
                        df["model"] = model_name
                        df["snr_db"] = snr if snr is not None else "clean"
                        df["dataset"] = dataset_name
                        all_rows.append(df)
                    else:
                        metrics = self.evaluate_aggregate_metrics(
                            model, loader, has_inclusions
                        )
                        row = {
                            "model": model_name,
                            "snr_db": snr if snr is not None else "clean",
                            "dataset": dataset_name,
                            **metrics
                        }
                        all_rows.append(pd.DataFrame([row]))
                
                print(f"  ✓ Completed {model_name}")
        
        return pd.concat(all_rows, ignore_index=True)


# ==================== USAGE EXAMPLE ====================

def main():
    # Initialize evaluator
    evaluator = ModelEvaluator(device="cuda")
    
    # ---- Dynamically extract RUN_IDS from Slurm report files ----
    import glob
    slurm_dir = os.path.join(os.getcwd(), 'Slurm_Training')
    run_pattern = re.compile(r'(run_\d{8}_\d{6})')
    report_to_run = {}
    for rpt in sorted(glob.glob(os.path.join(slurm_dir, 'Report-*.out'))):
        with open(rpt, 'r') as f:
            matches = run_pattern.findall(f.read())
        report_to_run[os.path.basename(rpt)] = matches[-1] if matches else None
    
    RUN_IDS = sorted(set(rid for rid in report_to_run.values() if rid is not None))
    
    print(f"Extracted {len(RUN_IDS)} unique run IDs from {len(report_to_run)} report files:")
    for rpt, rid in sorted(report_to_run.items()):
        print(f"  {rpt:45s} -> {rid or '(no ML run)'}")
    print()
    
    weight_paths = evaluator.find_weights_by_runs(RUN_IDS)
    
    # Load models
    models = evaluator.load_models(weight_paths)
    
    # Define test configurations
    base_test_dir = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Data"
    snr_levels = [30, 25, 20, 15]
    
    test_configs = [
        # Main test set
        {
            "test_dir": f"{base_test_dir}/Test/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": False,
            "dataset_name": "Main_Test"
        },
        # Inclusion datasets
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_3125cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.3125cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_625cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.625cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R1_25cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R1.25cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R2_5cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R2.5cm"
        }
    ]
    
    # Run evaluations
    print("\n" + "="*60)
    print("AGGREGATE METRICS EVALUATION")
    print("="*60)
    df_aggregate = evaluator.run_full_evaluation(
        models, test_configs, per_sample=False
    )
    
    print("\n" + "="*60)
    print("PER-SAMPLE METRICS EVALUATION")
    print("="*60)
    df_per_sample = evaluator.run_full_evaluation(
        models, test_configs, per_sample=True
    )
    
    # Save results
    # Create timestamped folder
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path("evaluation_results") / timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create subfolder for plots
    plots_dir = output_dir / "plots"
    plots_dir.mkdir(exist_ok=True)

    # Save CSVs
    df_aggregate.to_csv(output_dir / "aggregate_metrics.csv", index=False)
    df_per_sample.to_csv(output_dir / "per_sample_metrics.csv", index=False)

#     # Save styled Excel with highlighted best values
#     styled_df = highlight_best_per_snr_dataset(df_aggregate)
#     styled_df.to_excel(
#         output_dir / "aggregate_highlighted.xlsx",
#         engine="openpyxl",
#         index=False
#     )

    # Save summary of best models per metric
    summary = display_best_metrics_table(df_aggregate)
    summary.to_csv(output_dir / "best_models_summary.csv", index=False)
    
    # Generate and save plots for each dataset
    print("\nGenerating plots...")
    datasets = df_aggregate["dataset"].unique()

    for dataset in datasets:
        print(f"  Plotting {dataset}...")
        dataset_safe = dataset.replace("/", "_").replace(" ", "_").replace('.','_')

        # Determine metrics (add CNR if inclusions)
        metrics = ["mae", "rmse", "ssim"]
        has_cnr = "Inclusion" in dataset and "cnr" in df_aggregate.columns
        if has_cnr:
            metrics.append("cnr")

        # ========== COMBINED HEATMAP FIGURE ==========
        n_metrics = len(metrics)
        fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
        if n_metrics == 1:
            axes = axes.reshape(1, -1)

        for idx, metric in enumerate(metrics):
            # Get data
            plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()

            # Determine if metric has k/mu variants or is standalone (CNR)
            has_k = f"{metric}_k" in plot_df.columns
            has_mu = f"{metric}_mu" in plot_df.columns

            if has_k and has_mu:
                # k heatmap
                pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_k")
                sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="magma")
                axes[idx, 0].set_title(f"(k) {metric.upper()}")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel("Model")

                # mu heatmap
                pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_mu")
                sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[idx, 1], cmap="magma")
                axes[idx, 1].set_title(f"(mu) {metric.upper()}")
                axes[idx, 1].set_xlabel("SNR (dB)")
                axes[idx, 1].set_ylabel("")
            else:
                # CNR or other standalone metric
                pivot = plot_df.pivot(index="model", columns="snr_db", values=metric)
                sns.heatmap(pivot, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="viridis")
                axes[idx, 0].set_title(f"{metric.upper()}")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel("Model")

                # Hide second subplot
                axes[idx, 1].axis('off')

        plt.suptitle(f"{dataset} - Heatmaps", fontsize=16, y=1.002)
        plt.tight_layout()
        plt.savefig(plots_dir / f"{dataset_safe}_all_heatmaps.png", dpi=300, bbox_inches='tight')
        plt.close()

        # ========== COMBINED LINE PLOT FIGURE ==========
        fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
        if n_metrics == 1:
            axes = axes.reshape(1, -1)

        plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()
        plot_df["snr_numeric"] = plot_df["snr_db"].apply(lambda x: 999 if x == "clean" else float(x))
        plot_df["snr_label"] = plot_df["snr_db"].apply(lambda x: "Clean" if x == "clean" else str(x))
        plot_df = plot_df.sort_values("snr_numeric")

        for idx, metric in enumerate(metrics):
            has_k = f"{metric}_k" in plot_df.columns
            has_mu = f"{metric}_mu" in plot_df.columns

            if has_k and has_mu:
                # k line plot
                sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_k", 
                            hue="model", marker="o", style="model", ax=axes[idx, 0])
                unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
                axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 0].set_title(f"(k) {metric.upper()} vs SNR")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel(metric.upper())
                axes[idx, 0].grid(True)
                axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

                # mu line plot
                sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_mu",
                            hue="model", marker="o", style="model", ax=axes[idx, 1])
                axes[idx, 1].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 1].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 1].set_title(f"(mu) {metric.upper()} vs SNR")
                axes[idx, 1].set_xlabel("SNR (dB)")
                axes[idx, 1].set_ylabel(metric.upper())
                axes[idx, 1].grid(True)
                axes[idx, 1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            else:
                # CNR or standalone
                sns.lineplot(data=plot_df, x="snr_numeric", y=metric,
                            hue="model", marker="o", style="model", ax=axes[idx, 0])
                unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
                axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 0].set_title(f"{metric.upper()} vs SNR")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel(metric.upper())
                axes[idx, 0].grid(True)
                axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

                # Hide second subplot
                axes[idx, 1].axis('off')

        plt.suptitle(f"{dataset} - Metrics vs SNR", fontsize=16, y=1.002)
        plt.tight_layout()
        plt.savefig(plots_dir / f"{dataset_safe}_all_lineplot.png", dpi=300, bbox_inches='tight')
        plt.close()

        # ========== COMBINED BOX PLOT FIGURE (PER-SAMPLE) ==========
        print(f"    Generating per-sample distributions...")
        box_metrics = ["mae_k", "rmse_k", "mae_mu", "rmse_mu", "ssim_k", "ssim_mu"]
        if has_cnr:
            box_metrics.append("cnr")

        # Filter available metrics
        available_metrics = [m for m in box_metrics if m in df_per_sample.columns]

        if available_metrics:
            n_box = len(available_metrics)
            fig, axes = plt.subplots(n_box, 1, figsize=(14, 4*n_box))
            if n_box == 1:
                axes = [axes]

            plot_df_sample = df_per_sample[df_per_sample["dataset"] == dataset].copy()

            for idx, metric in enumerate(available_metrics):
                sns.boxplot(data=plot_df_sample, x="snr_db", y=metric, 
                           hue="model", palette="Set2", ax=axes[idx])
                axes[idx].set_title(f"Per-sample {metric} distribution")
                axes[idx].set_xlabel("SNR (dB)")
                axes[idx].set_ylabel(metric)
                axes[idx].legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

            plt.suptitle(f"{dataset} - Per-Sample Distributions", fontsize=16, y=1.002)
            plt.tight_layout()
            plt.savefig(plots_dir / f"{dataset_safe}_all_boxplots.png", dpi=300, bbox_inches='tight')
            plt.close()

    print("  ✓ All plots saved")

    # Save metadata
    metadata = {
        "timestamp": timestamp,
        "run_ids": RUN_IDS,
        "report_to_run_mapping": {k: v for k, v in report_to_run.items()},
        "num_models": len(models),
        "datasets": df_aggregate["dataset"].unique().tolist(),
        "snr_levels": df_aggregate["snr_db"].unique().tolist(),
        "total_samples_evaluated": len(df_per_sample),
        "models_evaluated": list(models.keys())
    }

    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("\n" + "="*60)
    print("✅ EVALUATION COMPLETE")
    print("="*60)
    print(f"Results saved to: {output_dir}/")
    print(f"  📁 {output_dir.name}/")
    print(f"     ├── aggregate_metrics.csv")
    print(f"     ├── per_sample_metrics.csv")
    print(f"     ├── aggregate_highlighted.xlsx")
    print(f"     ├── best_models_summary.csv")
    print(f"     ├── metadata.json")
    print(f"     └── plots/ ({len(list(plots_dir.glob('*.png')))} images)")
    print("="*60)
    
    
    return df_aggregate, df_per_sample


if __name__ == "__main__":
    df_agg, df_sample = main()



Using 9 run IDs extracted from Slurm report files
Matched weights.pth files: 11
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/mse/lam_data1_0/lam_phys0_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/gradnorm/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/pcgrad/lam_data1_0/lam_phys1_0/bs64/weights

/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2


  ✓ Completed FDTDNet/mse/lam_data1_0/lam_phys0_0/bs64


/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:269: RuntimeWarning: divide by zero encountered in divide
  S = (A1 * A2) / D
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:269: RuntimeWarning: invalid value encountered in divide
  S = (A1 * A2) / D
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/numpy/core/_methods.py:118: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/ski

  ✓ Completed FDTDNet/residual/het/gradnorm/lam_data1_0/lam_phys1_0/bs64


...

/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2


  ✓ Completed FDTDNet/residual/hom/pcgrad/lam_data1_0/lam_phys1_0/bs64

Evaluating on: Inclusion_R0.625cm


/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2


  ✓ Completed FDTDNet/mse/lam_data1_0/lam_phys0_0/bs64


/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:268: RuntimeWarning: overflow encountered in multiply
  D = B1 * B2
